# Lesson 8B: Recommender Systems Practical

## Introduction

In Lesson 8A, we built a recommender system from scratch. Now let's use production-ready libraries to build real-world recommendation engines.

We'll use:
- **Surprise**: Scikit-learn-style library for recommender systems
- **Implicit**: Fast library for implicit feedback (clicks, views)
- **TensorFlow/PyTorch**: Deep learning approaches

In this lesson:
1. Load real MovieLens data
2. Build recommendations with Surprise (SVD, SVD++, NMF)
3. Handle implicit feedback with Implicit library
4. Implement deep learning content-based filtering
5. Evaluate recommendation quality
6. Deploy a recommendation API

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [MovieLens Dataset](#movielens-dataset)
4. [Surprise Library](#surprise-library)
5. [Comparing Algorithms](#comparing-algorithms)
6. [Implicit Feedback](#implicit-feedback)
7. [Deep Learning Approach](#deep-learning)
8. [Evaluation Metrics](#evaluation-metrics)
9. [Production Deployment](#production-deployment)
10. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

| Library | Purpose |
|---------|--------|
| NumPy | Numerical computing |
| Pandas | Data manipulation |
| Surprise | Recommender algorithms |
| Implicit | Implicit feedback |
| PyTorch | Deep learning |
| Scikit-learn | Metrics |

In [ ]:
# Install Surprise and Implicit if not available
# !pip install scikit-surprise implicit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple

# Surprise library
from surprise import Dataset, Reader, SVD, SVDpp, NMF
from surprise.model_selection import cross_validate, train_test_split
from surprise import accuracy

# Implicit library
import implicit
from scipy.sparse import csr_matrix

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

np.random.seed(42)
torch.manual_seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="movielens-dataset"></a>
## MovieLens Dataset

We'll use the MovieLens 100K dataset:
- 100,000 ratings
- 943 users  
- 1,682 movies
- Ratings: 1-5 stars

Surprise library provides easy access to this dataset.

In [ ]:
# Load MovieLens 100k dataset
data = Dataset.load_builtin('ml-100k')
print("✅ MovieLens 100k dataset loaded")

# Convert to DataFrame for analysis
df = pd.DataFrame(data.raw_ratings, columns=['user', 'item', 'rating', 'timestamp'])
df['rating'] = df['rating'].astype(float)

print(f"\nDataset shape: {df.shape}")
print(f"Number of users: {df['user'].nunique():,}")
print(f"Number of items: {df['item'].nunique():,}")
print(f"Number of ratings: {len(df):,}")
print(f"\nRating distribution:")
print(df['rating'].value_counts().sort_index())

# Display sample
df.head(10)

In [ ]:
# Visualize rating distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating histogram
axes[0].hist(df['rating'], bins=5, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Rating', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Distribution of Ratings', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Ratings per user
ratings_per_user = df.groupby('user').size()
axes[1].hist(ratings_per_user, bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Number of Ratings', fontsize=12)
axes[1].set_ylabel('Number of Users', fontsize=12)
axes[1].set_title('Ratings per User', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nAverage ratings per user: {ratings_per_user.mean():.1f}")
print(f"Median ratings per user: {ratings_per_user.median():.1f}")

<a name="surprise-library"></a>
## Surprise Library: SVD

Singular Value Decomposition (SVD) is one of the most popular matrix factorization techniques.

$$\hat{r}_{ui} = \mu + b_u + b_i + q_i^T p_u$$

Where:
- $\mu$ = global mean rating
- $b_u$ = user bias
- $b_i$ = item bias  
- $q_i$ = item factors
- $p_u$ = user factors

In [ ]:
# Split data
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# Train SVD model
print("Training SVD model...")
svd = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd.fit(trainset)

# Make predictions
predictions = svd.test(testset)

# Evaluate
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

print(f"\n✅ SVD Model Performance:")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")

In [ ]:
# Get recommendations for a specific user
def get_top_n_recommendations(predictions, n=10):
    """Get top-N recommendations for each user."""
    top_n = {}
    
    for uid, iid, true_r, est, _ in predictions:
        if uid not in top_n:
            top_n[uid] = []
        top_n[uid].append((iid, est))
    
    # Sort and get top N
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    
    return top_n

# Get recommendations
top_n = get_top_n_recommendations(predictions, n=10)

# Show recommendations for first user
first_user = list(top_n.keys())[0]
print(f"Top 10 recommendations for user '{first_user}':")
for i, (movie_id, predicted_rating) in enumerate(top_n[first_user], 1):
    print(f"{i:2d}. Movie {movie_id}: Predicted rating = {predicted_rating:.2f}")

<a name="comparing-algorithms"></a>
## Comparing Algorithms

Let's compare different matrix factorization algorithms:
- **SVD**: Basic matrix factorization
- **SVD++**: Incorporates implicit feedback
- **NMF**: Non-Negative Matrix Factorization

In [ ]:
# Define algorithms to compare
algorithms = {
    'SVD': SVD(n_factors=100, n_epochs=20, random_state=42),
    'SVD++': SVDpp(n_factors=20, n_epochs=10, random_state=42),  # Fewer factors due to computational cost
    'NMF': NMF(n_factors=50, n_epochs=20, random_state=42)
}

results = []

for name, algorithm in algorithms.items():
    print(f"\nEvaluating {name}...")
    
    # Cross-validation
    cv_results = cross_validate(algorithm, data, measures=['RMSE', 'MAE'], cv=5, verbose=False)
    
    results.append({
        'Algorithm': name,
        'RMSE': cv_results['test_rmse'].mean(),
        'MAE': cv_results['test_mae'].mean(),
        'Fit Time': cv_results['fit_time'].mean(),
        'Test Time': cv_results['test_time'].mean()
    })

# Display results
results_df = pd.DataFrame(results)
print("\n" + "="*70)
print("ALGORITHM COMPARISON")
print("="*70)
print(results_df.to_string(index=False))
print("\n✅ Best RMSE: " + results_df.loc[results_df['RMSE'].idxmin(), 'Algorithm'])

<a name="implicit-feedback"></a>
## Implicit Feedback

Not all feedback is explicit ratings. Often we have:
- Clicks (user clicked on a product)
- Views (user watched a video)
- Purchases (user bought an item)
- Time spent (user spent 10 minutes reading)

These are **implicit feedback** - no explicit rating, but signal of interest.

The **Implicit** library implements Alternating Least Squares (ALS) optimized for implicit feedback.

In [ ]:
# Convert explicit ratings to implicit feedback (binary)
# Assume rating >= 4 means "liked"
implicit_df = df.copy()
implicit_df['confidence'] = (implicit_df['rating'] >= 4).astype(int)

# Create user-item matrix
user_ids = implicit_df['user'].astype('category').cat.codes
item_ids = implicit_df['item'].astype('category').cat.codes
confidence = implicit_df['confidence'].values

# Build sparse matrix
user_item_matrix = csr_matrix((confidence, (user_ids, item_ids)))

print(f"User-item matrix shape: {user_item_matrix.shape}")
print(f"Sparsity: {1 - user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.4f}")

In [ ]:
# Train ALS model
print("Training ALS model for implicit feedback...")
als_model = implicit.als.AlternatingLeastSquares(
    factors=50,
    regularization=0.01,
    iterations=15,
    random_state=42
)

als_model.fit(user_item_matrix)
print("✅ ALS model trained!")

# Get recommendations for a user
user_id = 0
recommendations = als_model.recommend(user_id, user_item_matrix[user_id], N=10)

print(f"\nTop 10 recommendations for user {user_id}:")
for i, (item_idx, score) in enumerate(recommendations, 1):
    print(f"{i:2d}. Item {item_idx}: Score = {score:.4f}")

<a name="deep-learning"></a>
## Deep Learning Approach: Neural Collaborative Filtering

Use neural networks to learn non-linear interactions between users and items.

In [ ]:
class NeuralCollaborativeFiltering(nn.Module):
    """Neural network for collaborative filtering."""
    
    def __init__(self, n_users: int, n_items: int, embedding_dim: int = 50, hidden_dims: List[int] = [128, 64]):
        super().__init__()
        
        # Embeddings
        self.user_embedding = nn.Embedding(n_users, embedding_dim)
        self.item_embedding = nn.Embedding(n_items, embedding_dim)
        
        # MLP layers
        layers = []
        input_dim = embedding_dim * 2
        
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.2))
            input_dim = hidden_dim
        
        layers.append(nn.Linear(input_dim, 1))
        
        self.mlp = nn.Sequential(*layers)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
    
    def forward(self, user_ids: torch.Tensor, item_ids: torch.Tensor) -> torch.Tensor:
        # Get embeddings
        user_emb = self.user_embedding(user_ids)
        item_emb = self.item_embedding(item_ids)
        
        # Concatenate
        x = torch.cat([user_emb, item_emb], dim=1)
        
        # Pass through MLP
        output = self.mlp(x)
        
        return output.squeeze()

print("✅ Neural Collaborative Filtering model defined!")

In [ ]:
# Prepare data for PyTorch
n_users = df['user'].nunique()
n_items = df['item'].nunique()

# Map users and items to indices
user_to_idx = {user: idx for idx, user in enumerate(df['user'].unique())}
item_to_idx = {item: idx for idx, item in enumerate(df['item'].unique())}

df['user_idx'] = df['user'].map(user_to_idx)
df['item_idx'] = df['item'].map(item_to_idx)

# Convert to tensors
user_ids = torch.LongTensor(df['user_idx'].values)
item_ids = torch.LongTensor(df['item_idx'].values)
ratings = torch.FloatTensor(df['rating'].values)

# Train/test split
train_size = int(0.8 * len(df))
indices = torch.randperm(len(df))
train_indices = indices[:train_size]
test_indices = indices[train_size:]

print(f"Training samples: {len(train_indices):,}")
print(f"Test samples: {len(test_indices):,}")

In [ ]:
# Initialize model
model = NeuralCollaborativeFiltering(n_users, n_items, embedding_dim=50, hidden_dims=[128, 64])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
n_epochs = 10
batch_size = 1024
train_losses = []

print("Training Neural Collaborative Filtering...\n")

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0
    
    # Mini-batch training
    for i in range(0, len(train_indices), batch_size):
        batch_indices = train_indices[i:i+batch_size]
        
        batch_users = user_ids[batch_indices]
        batch_items = item_ids[batch_indices]
        batch_ratings = ratings[batch_indices]
        
        # Forward pass
        predictions = model(batch_users, batch_items)
        loss = criterion(predictions, batch_ratings)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    # Evaluate on test set
    model.eval()
    with torch.no_grad():
        test_predictions = model(user_ids[test_indices], item_ids[test_indices])
        test_loss = criterion(test_predictions, ratings[test_indices])
    
    avg_train_loss = epoch_loss / (len(train_indices) / batch_size)
    train_losses.append(avg_train_loss)
    
    print(f"Epoch {epoch+1}/{n_epochs}: Train Loss = {avg_train_loss:.4f}, Test Loss = {test_loss:.4f}")

print("\n✅ Training complete!")

<a name="evaluation-metrics"></a>
## Evaluation Metrics

### Rating Prediction Metrics:
- **RMSE**: Root Mean Squared Error
- **MAE**: Mean Absolute Error

### Ranking Metrics:
- **Precision@K**: Of top K recommendations, how many are relevant?
- **Recall@K**: Of all relevant items, how many are in top K?
- **NDCG**: Normalized Discounted Cumulative Gain (considers ranking order)
- **MAP**: Mean Average Precision

<a name="production-deployment"></a>
## Production Deployment

### Best Practices:

1. **Offline Training**:
   - Train models periodically (daily/weekly)
   - Store learned embeddings

2. **Online Serving**:
   - Precompute recommendations for active users
   - Use approximate nearest neighbor search for scalability
   - Cache popular recommendations

3. **A/B Testing**:
   - Compare recommendation algorithms
   - Measure click-through rate, conversion rate

4. **Monitoring**:
   - Track recommendation coverage
   - Monitor for popularity bias
   - Detect and handle cold start

5. **Ethical Considerations**:
   - Avoid filter bubbles
   - Provide diversity in recommendations
   - Give users control (explain, allow feedback)

<a name="conclusion"></a>
## Conclusion

### What We Built

1. **Surprise library**: Production SVD, SVD++, NMF implementations
2. **Implicit library**: Fast ALS for implicit feedback
3. **Deep learning**: Neural collaborative filtering with PyTorch
4. **Evaluation**: Proper metrics for recommender systems

### Algorithm Comparison

| Algorithm | Best For | Pros | Cons |
|-----------|----------|------|------|
| SVD | Explicit ratings | Fast, interpretable | Linear only |
| SVD++ | Explicit + implicit | Incorporates both signals | Slower training |
| ALS | Implicit feedback | Very fast, scalable | Explicit ratings only |
| Neural CF | Complex patterns | Learns non-linearities | Requires more data |

### When to Use What

- ✅ **SVD/NMF**: Explicit ratings (1-5 stars), need fast training
- ✅ **ALS**: Implicit feedback (clicks, views), large scale
- ✅ **Neural CF**: Lots of data, complex interaction patterns
- ✅ **Hybrid**: Combine collaborative + content-based for cold start

### Next Steps

1. Try different hyperparameters
2. Implement content-based features
3. Build a hybrid system
4. Deploy as an API
5. Run A/B tests

Recommender systems power much of the internet - you now have the tools to build them! 🎬🎵🛍️